# Solution: Prophet as an Auto-Tuned Classical Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet

In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
US = {"Orange": "#EE7622"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
DATA_PATH = "../data/electricity_demand.csv"

The electricity demand data has a seasonal pattern and a temperature regressor. Fit Prophet with the regressor and compare the forecast quality to the same train/test split.

In [ ]:
# Load data and split
df = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").asfreq("MS")
train = df.loc[:"2023-12-01"].reset_index()
test = df.loc["2024-01-01":].reset_index()

# Prophet requires columns named ds and y
prophet_train = train.rename(columns={"date": "ds", "demand_mwh": "y"})

In [ ]:
# Instantiate Prophet and add temperature regressor
m = Prophet()
m.add_regressor("avg_temp_f")
m.fit(prophet_train)

In [ ]:
# Build future dataframe for the test period and merge temperature
future = m.make_future_dataframe(periods=12, freq="MS")
future = future.merge(df[["avg_temp_f"]].reset_index(), left_on="ds", right_on="date", how="left")
forecast = m.predict(future)

In [ ]:
# Extract the test-period predictions and compute errors
preds = forecast.set_index("ds").loc[test["date"], "yhat"]
actuals = test.set_index("date")["demand_mwh"]
mae = np.mean(np.abs(preds - actuals))
mape = np.mean(np.abs((preds - actuals) / actuals)) * 100
print(f"MAE: {mae:.1f} MW")
print(f"MAPE: {mape:.1f}%")

In [ ]:
# Plot the Prophet forecast components
fig = m.plot_components(forecast)
plt.show()

In [ ]:
# Plot actual vs predicted for the test period
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["date"], train["demand_mwh"], color=UN["Black"], label="Train")
ax.plot(test["date"], actuals, color=UB["Brand Blue"], label="Actual")
ax.plot(test["date"], preds, color=US["Orange"], label="Prophet")
ax.axvline(train["date"].iloc[-1], color=UN["Dark Gray"], linestyle="--")
ax.set_title("Prophet Forecast vs Actual", fontsize=18, fontweight="bold")
ax.set_ylabel("Demand (MWh)", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend()
plt.tight_layout()
plt.show()